# Backprop oracle
In this notebook, I want to implement a scalar-level class that can model a Neural Network. Similar to Andrej Karpathy's series, we instantiate the class when we define our forward pass, and the class variables keep track of the backward pass for us. 

We then use this code as an oracle to verify the correctness of hand-derived backward passes

In [14]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('fivethirtyeight')
sns.set_context('notebook')

In [15]:
from functools import total_ordering
@total_ordering
class Scalar:
    def __init__(self, value, up_nodes=None, no_grad=False):
        self.value = np.float64(value)
        self.grad = np.float64(0)
        self.up = up_nodes or []
        self.no_grad = no_grad


    def __repr__(self):
        return str(f'Sc.{self.value:.3f}')
    
    
    def __hash__(self):
        return id(self)


    def __lt__(self, other): 
        other = other if isinstance(other, Scalar) else Scalar(other, no_grad=True)
        return self.value < other.value
    

    def __eq__(self, other):
        other = other if isinstance(other, Scalar) else Scalar(other, no_grad=True)
        return self.value == other.value
    

    def __add__(self, other):
        other = other if isinstance(other, Scalar) else Scalar(other, no_grad=True)
        nodes = []
        for node in (self, other):
            if not node.no_grad:
                nodes.append((node, 1))
        return Scalar(self.value + other.value, nodes)
    
    
    def __mul__(self, other):
        other = other if isinstance(other, Scalar) else Scalar(other, no_grad=True)
        nodes = []
        for node in (self, other):
            if not node.no_grad:
                nodes.append((node, other.value if node is self else self.value))
        return Scalar(self.value * other.value, nodes)
    

    def __pow__(self, other):
        other = other if isinstance(other, Scalar) else Scalar(other, no_grad=True)
        nodes = []
        if not self.no_grad:
            deriv = other.value * (self.value ** (other.value -1))
            nodes.append((self, deriv))
        if not other.no_grad:
            deriv = (self.value ** other.value) * np.log(self.value)
            nodes.append((other, deriv))
        return Scalar(self.value ** other.value, nodes)       


    def __rpow__(self, other): 
        other = other if isinstance(other, Scalar) else Scalar(other, no_grad=True)
        return other ** self
    

    def log(self):
        nodes = []
        if not self.no_grad:
            deriv = 1 / self.value
            nodes.append((self, deriv))
        return Scalar(np.log(self.value), nodes)
        
        
    def __radd__(self, other): return self.__add__(other)
    def __neg__(self): return self.__mul__(-1)
    def __rmul__(self, other): return self.__mul__(other)
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return other + (-self)
    def __truediv__(self, other): return self * other**np.float64(-1)
    def __rtruediv__(self, other): return (self**np.float64(-1)) * other


    def node_crawl(self, node_list, visited): # nodes ordered upstream first
        for node in self.up:
            if node[0] not in visited: node[0].node_crawl(node_list, visited)
        node_list.append(self)
        visited.add(self)
        self.grad = 0.0 # reset grad to zero before backprop
        

    def backward(self, first_node=True):
        if first_node == True:
            node_list = []
            visited = set()
            self.node_crawl(node_list, visited)
            self.grad = 1
            for node in reversed(node_list):
                node.backward(first_node=False)
            return
            
        for node, derivative in self.up:
            node.grad += self.grad * derivative


In [16]:
def init(*shape, scalar=False, type='normal'):
    if type == 'int':
        vals = rng.randint(0, 10, size=shape)
    elif type == 'bool':
        vals = rng.randint(0, 2, size=shape)
    else:
        vals = rng.normal(0, 0.1, size=shape)
    if scalar == True:
        return np.vectorize(Scalar)(vals)
    else:
        return vals

s_log = np.vectorize(Scalar.log)
rng = np.random.default_rng(167)

## Parse your code here
Positions are labeled "HERE"

In [17]:
# Leave any hyperparameters you need HERE:
BATCH_SIZE = 128 # how many examples to process at once
EMB_D = 4 # number of embedding dimensions
CONTEXT_SIZE = 3 # how many characters the model sees at once
HIDDEN_D = 300 # number of neurons in the hidden layer
dict_size = 27

# define batch & ground_truth dimensions HERE:
batch = np.random.randint(0, 26, (128, 3, 27))
Y = np.random.randint(0, 2, size=BATCH_SIZE)

# initialize parameters HERE:
W_emb = rng.normal(0, 1, size=(dict_size, EMB_D))
W1 = rng.normal(0, (CONTEXT_SIZE * EMB_D)**-0.5, size=(CONTEXT_SIZE * EMB_D, HIDDEN_D))
b1 = rng.uniform(size=(1, W1.shape[1]))
W2 = rng.normal(0, (2/HIDDEN_D)**0.5, size=(HIDDEN_D, dict_size))
b2 = rng.uniform(size=(1, W2.shape[1]))


def scalar_variables():
    # convert all parameters to scalars HERE for the scalar pass:
    global W_emb, W1, b1, W2, b2
    W_emb = np.vectorize(Scalar)(W_emb)
    W1 = np.vectorize(Scalar)(W1)
    b1 = np.vectorize(Scalar)(b1)
    W2 = np.vectorize(Scalar)(W2)
    b2 = np.vectorize(Scalar)(b2)


In [18]:
# Normal forward/backward pass HERE:
# Forward pass:
X0 = (batch @ W_emb).reshape(BATCH_SIZE, CONTEXT_SIZE * EMB_D)
H1 = X0 @ W1 + b1
X1 = np.where(H1 > 0, H1, H1 * 0.01)
H2 = X1 @ W2 + b2
H2_norm = H2 - np.max(H2, axis=1, keepdims=True)
Y_hat = np.e ** H2_norm / np.sum(np.e**(H2_norm), axis=1, keepdims=True)
loss = np.mean(-np.log(Y_hat[np.arange(BATCH_SIZE), Y]))

# Backward Pass:
dH2 = Y_hat.copy()
dH2[np.arange(BATCH_SIZE), Y] -= 1
dH2 /= BATCH_SIZE

dX1 = dH2 @ W2.T
dW2 = X1.T @ dH2
db2 = dH2.sum(axis=0, keepdims=True)

dH1 = np.where(H1 > 0, dX1, 0.01 * dX1)

dX0 = dH1 @ W1.T
dW1 = X0.T @ dH1
db1 = dH1.sum(axis=0, keepdims=True)
dW_emb = (batch.transpose((0, 2, 1)) @ dX0.reshape(BATCH_SIZE, CONTEXT_SIZE, EMB_D)).sum(axis=0)

In [ ]:
# Scalar forward pass:
scalar_variables()

# Forward pass HERE:
X0 = (batch @ W_emb).reshape(BATCH_SIZE, CONTEXT_SIZE * EMB_D)
H1 = X0 @ W1 + b1
X1 = np.where(H1 > 0, H1, H1 * 0.01)
H2 = X1 @ W2 + b2
H2_norm = H2 - np.max(H2, axis=1, keepdims=True)
Y_hat = np.e ** H2_norm / np.sum(np.e**(H2_norm), axis=1, keepdims=True)
scalar_loss = np.mean(-s_log(Y_hat[np.arange(BATCH_SIZE), Y]))
#####

scalar_loss.backward()

print(f'Loss: {loss:.3f}')
print(f'Scalar loss: {scalar_loss.value:.3f}')


Loss: 114.680
Scalar loss: 114.680


In [ ]:
# Verification (add in grads HERE):
grads = [dW_emb, dW1, db1, dW2, db2]
scalar_grads = [np.vectorize(lambda s: s.grad)(m) for m in [W_emb, W1, b1, W2, b2]]
#####

for pair in zip(grads, scalar_grads):
    print(np.allclose(pair[0], pair[1])) # Check if all tensors are close in values



True
True
True
True
True
